In [ ]:
if 'google.colab' in str(get_ipython()):
    ! git clone -b main https://github.com/edsonportosilva/OptiCommPy
    from os import chdir as cd
    cd('/content/OptiCommPy/')
    ! pip install .

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
import cupy as cp;
from cupy.lib.stride_tricks import as_strided
from numpy.fft import fft, ifft, fftfreq
from  scipy.constants import c
import matplotlib.pyplot as plt
import time

from optic.comm.modulation import modulateGray, grayMapping
from optic.dsp.core import pnorm, upsample, firFilter, pulseShape, signalPower, phaseNoise, decimate, symbolSync, finddelay
from optic.dsp.equalization import edc, mimoAdaptEqualizer
from optic.models.devices import iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel, mzm, edfa
from optic.models.channels import linearFiberChannel, awgn
from optic.utils import parameters
from optic.comm.metrics import fastBERcalc
from optic.plot import pconst, eyediagram, plotPSD
from optic.models.tx import simpleWDMTx
from optic.dsp.carrierRecovery import cpr

/usr/local/lib/python3.12/dist-packages/cupyx/jit/_interface.py:173: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


In [ ]:
def create_windows(x, paramEq):
    """
    Cria janelas temporais para polarização X e Y (coluna 0 e coluna 1)
    """
    N = x.shape[0] - paramEq.nTaps + 1
    shape = (N, paramEq.nTaps)
    x0 = cp.ascontiguousarray(x[:, 0])                                          # Colocando x[:, 0] como sinal contíguos
    x1 = cp.ascontiguousarray(x[:, 1])                                          # Colocando x[:, 1] como sinal contíguos

    stridesx = x0.itemsize
    xV_ = as_strided(x0, shape=shape, strides=(stridesx,stridesx))
    xV = xV_[:, ::-1]                                                           # Polarização X é a coluna 0

    stridesy = x1.itemsize
    xH_ = as_strided(x1, shape=shape, strides=(stridesy,stridesy))
    xH = xH_[:, ::-1]                                                           # Polarização Y é a coluna 1

    return xV, xH

In [ ]:
def ddlmsUp(x, constSymb, nModes, paramEq):
    """
    Equalizador DD-LMS para um sistema MIMO 2x2.
    Implementação amostra-por-amostra com lógica de atualização corrigida.
    """

    # Parâmetros para o processamento em blocos
    blockSize = 2**11
    N = x.shape[0] - paramEq.nTaps + 1
    nBlocks = N // blockSize

    # Obtém o atraso da filtragem FIR
    delay = (paramEq.nTaps - 1) // 2

    # --- Pré-alocação de Memória ---
    y = cp.zeros((len(x), nModes), dtype=cp.complex64)
    e = cp.zeros((len(x), nModes), dtype=cp.complex64)
    w = cp.zeros((paramEq.nTaps, nModes**2), dtype=cp.complex64)

    # single spike initialization
    w[delay, 0] = 1                                                             # Filtro principal w_xx (para o Mode 0)
    w[delay, 3] = 1                                                             # Filtro principal w_yy (para o Mode 1)

    for i in tqdm(range(nBlocks), disable=not(paramEq.progBar)):

        n = i * blockSize
        n_delay = delay + n + blockSize
        x_ = x[n : n + blockSize + paramEq.nTaps - 1, :]

        # cria janelas temporais para polarização X e Y (coluna 0 e coluna 1)
        xV, xH = create_windows(x_, paramEq)

        # calcula a saída do equalizador 2x2
        y[delay + n : n_delay, 0] = xV @ w[:, 0] + xH @ w[:, 1]
        y[delay + n : n_delay, 1] = xV @ w[:, 2] + xH @ w[:, 3]

        y0 = y[delay + n : n_delay, 0]
        y1 = y[delay + n : n_delay, 1]

        # decisão brusca
        d0 = constSymb[cp.argmin(cp.abs(y0[:, None] - constSymb), axis = 1)]
        d1 = constSymb[cp.argmin(cp.abs(y1[:, None] - constSymb), axis = 1)]

        e[delay + n : n_delay, 0] = d0 - y[delay + n : n_delay, 0]
        e[delay + n : n_delay, 1] = d1 - y[delay + n : n_delay, 1]

        # atualiza os coeficientes do filtro
        w[:,0] += paramEq.mu[0] * (xV.conj().T @ (e[delay + n : n_delay, 0])) / blockSize    # w_xx = w[:, 0]
        w[:,2] += paramEq.mu[0] * (xV.conj().T @ (e[delay + n : n_delay, 1])) / blockSize    # w_yx = w[:, 2]
        w[:,1] += paramEq.mu[0] * (xH.conj().T @ (e[delay + n : n_delay, 0])) / blockSize    # w_xy = w[:, 1]
        w[:,3] += paramEq.mu[0] * (xH.conj().T @ (e[delay + n : n_delay, 1])) / blockSize    # w_yy = w[:, 3]


    return y, e, w

In [ ]:
def rdeUp(x, constSymb, nModes, paramEq, y=None, e=None, w=None, preConv=False):
    """
    Radius-Directed Equalization Algorithm

    Returns
    -------
    tuple
        - y (cp.array): estimativa dos símbolos.
        - e (cp.array): erro associado a cada modo de polarização.
        - w (cp.array): matriz de coeficientes.

    Referências
    -----------
    [1] Digital Coherent Optical Systems, Architecture and Algorithms
    """

    # Parâmetros para o processamento em blocos
    blockSize = 2**11
    N = x.shape[0] - paramEq.nTaps + 1
    nBlocks = N // blockSize

    # Obtém o atraso da filtragem FIR
    delay = (paramEq.nTaps - 1) // 2

    # obtem os raios da constelação M-QAM
    Rrde = cp.unique(cp.abs(constSymb))

    if preConv == False:

        paramEq.N2 = 0

        y = cp.zeros((len(x), nModes), dtype=cp.complex64)
        e = cp.zeros((len(x), nModes), dtype=cp.complex64)
        w = cp.zeros((paramEq.nTaps, nModes**2), dtype=cp.complex64)

        # single spike initialization
        w[delay, 0] = 1                                                             # Filtro principal w_xx (para o Mode 0)
        w[delay, 3] = 1                                                             # Filtro principal w_yy (para o Mode 1)

    N1_block = paramEq.N1 // blockSize if paramEq.N1 > 0 else -1

    for i in tqdm(range(nBlocks), disable=not(paramEq.progBar)):

        n = i * blockSize
        n_delay = delay + n + blockSize
        x_ = x[n : n + blockSize + paramEq.nTaps - 1, :]

        # cria janelas temporais para polarização X e Y (coluna 0 e coluna 1)
        xV, xH = create_windows(x_, paramEq)

        # calcula a saída do equalizador 2x2
        y[delay + n : n_delay, 0] = xV @ w[:, 0] + xH @ w[:, 1]
        y[delay + n : n_delay, 1] = xV @ w[:, 2] + xH @ w[:, 3]

        y0 = cp.abs(y[delay + n : n_delay, 0])
        y1 = cp.abs(y[delay + n : n_delay, 1])

        R1 = cp.argmin(cp.abs(Rrde - y0[:, None]), axis = 1)
        R2 = cp.argmin(cp.abs(Rrde - y1[:, None]), axis = 1)

        # calcula e atualiza erro para cada modo de polarização
        e[delay + n : n_delay, 0] = y[delay + n : n_delay, 0] * (Rrde[R1]**2 - y0**2)
        e[delay + n : n_delay, 1] = y[delay + n : n_delay, 1] * (Rrde[R2]**2 - y1**2)

        # atualiza os coeficientes do filtro
        w[:,0] += paramEq.mu[0] * (xV.conj().T @ (e[delay + n : n_delay, 0])) / blockSize    # w_xx = w[:, 0]
        w[:,2] += paramEq.mu[0] * (xV.conj().T @ (e[delay + n : n_delay, 1])) / blockSize    # w_yx = w[:, 2]
        w[:,1] += paramEq.mu[0] * (xH.conj().T @ (e[delay + n : n_delay, 0])) / blockSize    # w_xy = w[:, 1]
        w[:,3] += paramEq.mu[0] * (xH.conj().T @ (e[delay + n : n_delay, 1])) / blockSize    # w_yy = w[:, 3]

        if i == N1_block:
            # Defina a polarização Y como ortogonal a X para evitar
            # a convergência para a mesma polarização (evitar a singularidade CMA)
            w[:,3] =  cp.conj(w[:,0][::-1])
            w[:,2] = -cp.conj(w[:,1][::-1])

    return y, e, w

In [ ]:
def cmaUp(x, constSymb, nModes, paramEq, preConv=False):
    """
    Constant-Modulus Algorithm

    Returns
    -------
    tuple
        - y (cp.array): estimativa dos símbolos.
        - e (cp.array): erro associado a cada modo de polarização.
        - w (cp.array): matriz de coeficientes.

    Referências
    -----------
    [1] Digital Coherent Optical Systems, Architecture and Algorithms
    """

    # Parâmetros para o processamento em blocos
    blockSize = 2**11
    N = x.shape[0] - paramEq.nTaps + 1
    blocks_default = N // blockSize

    # Obtém o atraso da filtragem FIR
    delay = (paramEq.nTaps - 1) // 2

    y = cp.zeros((len(x), nModes), dtype=cp.complex64)
    e = cp.zeros((len(x), nModes), dtype=cp.complex64)
    w = cp.zeros((paramEq.nTaps, nModes**2), dtype=cp.complex64)

    # single spike initialization
    w[delay, 0] = 1                                                             # Filtro principal w_xx (para o Mode 0)
    w[delay, 3] = 1                                                             # Filtro principal w_yy (para o Mode 1)

    # constante relacionada às características da modulação para o algoritmo CMA
    R = cp.mean(cp.abs(constSymb)**4) / cp.mean(cp.abs(constSymb)**2)

    N1_block = paramEq.N1 // blockSize if paramEq.N1 > 0 else -1
    nBlocks = paramEq.N2 // blockSize if preConv else blocks_default

    for i in tqdm(range(nBlocks), disable=not(paramEq.progBar)):

        n = i * blockSize
        n_delay = delay + n + blockSize
        x_ = x[n : n + blockSize + paramEq.nTaps - 1, :]

        # cria janelas temporais para polarização X e Y (coluna 0 e coluna 1)
        xV, xH = create_windows(x_, paramEq)

        # calcula a saída do equalizador 2x2
        y[delay + n : n_delay, 0] = xV @ w[:, 0] + xH @ w[:, 1]
        y[delay + n : n_delay, 1] = xV @ w[:, 2] + xH @ w[:, 3]

        # calcula e atualiza erro para cada modo de polarização
        e[delay + n : n_delay, 0] = y[delay + n : n_delay, 0] * (R - cp.abs(y[delay + n : n_delay, 0])**2)
        e[delay + n : n_delay, 1] = y[delay + n : n_delay, 1] * (R - cp.abs(y[delay + n : n_delay, 1])**2)

        # atualiza os coeficientes do filtro
        w[:,0] += paramEq.mu[0] * (xV.conj().T @ (e[delay + n : n_delay, 0])) / blockSize    # w_xx = w[:, 0]
        w[:,2] += paramEq.mu[0] * (xV.conj().T @ (e[delay + n : n_delay, 1])) / blockSize    # w_yx = w[:, 2]
        w[:,1] += paramEq.mu[0] * (xH.conj().T @ (e[delay + n : n_delay, 0])) / blockSize    # w_xy = w[:, 1]
        w[:,3] += paramEq.mu[0] * (xH.conj().T @ (e[delay + n : n_delay, 1])) / blockSize    # w_yy = w[:, 3]

        # Ortogonalização CMA apenas uma vez quando N1 for atingido
        if i == N1_block:
            w[:,3] =  cp.conj(w[:,0][::-1])
            w[:,2] = -cp.conj(w[:,1][::-1])

    if preConv:
        w_ = cp.max(cp.abs(w))
        w = w/w_ if w_ > 1e-6 else w
        y, e, w = rdeUp(x, constSymb, nModes, paramEq, y, e, w, preConv=True)

    return y, e, w

In [ ]:
def mimoAdaptEq(x, paramEq):
    """
    Equalizador adaptativo MIMO 2x2

    Returns
    -------
    tuple
        - y (cp.array): estimativa dos símbolos.
        - e (cp.array): erro associado a cada modo de polarização.
        - w (cp.array): matriz de coeficientes.

    Raises
    ------
    ValueError
        Caso o sinal não possua duas polarizações.

    ValueError
        Caso o algoritmo seja especificado incorretamente.
    """

    if x.shape[1] != 2:
        raise ValueError("O sinal deve conter duas polarizações")

    nModes = x.shape[1]

    # obtem os símbolos da constelação
    constSymb = grayMapping(paramEq.M, paramEq.constType)
    # normaliza os símbolos da constelação
    constSymb = pnorm(constSymb)
    constSymb = cp.asarray(constSymb)

    if paramEq.alg == 'cma':
        y, e, w = cmaUp(x, constSymb, nModes, paramEq)
    elif paramEq.alg == 'rde':
        y, e, w = rdeUp(x, constSymb, nModes, paramEq)
    elif paramEq.alg == 'cma-to-rde':
        y, e, w = cmaUp(x, constSymb, nModes, paramEq, preConv=True)
    elif paramEq.alg == 'dd-lms':
        y, e, w = ddlmsUp(x, constSymb, nModes, paramEq)
    else:
        raise ValueError("Algoritmo de equalização especificado incorretamente.")

    return y, e, w

In [ ]:
    ############## Canal Óptico ##############
def canal_optico(sigWDM_Tx, paramEDFA, paramTx):

  sigCh       = edfa(sigWDM_Tx, paramEDFA)                                          # modelo linear do EDFA

  # # deteriorates the signal-to-noise ratio before the fiber
  SNR         = 13
  sigAWGN     = awgn(sigWDM_Tx, SNR, Fs, paramTx.Rs)

  # simulate linear signal propagation
  sigCh       = linearFiberChannel(sigWDM_Tx, paramFiber)

  return sigCh

In [ ]:
    ############## Receptor Óptico (PMD)##############
def receptor_optico(sigCh, paramTx, paramPD):

  # parameters
  chIndex     = 0  # index of the channel to be demodulated

  Fc          = paramTx.Fc
  Ts          = 1/Fs
  freqGrid    = paramTx.wdmFreqGrid
  π           = np.pi
  t           = np.arange(0, len(sigCh))*Ts

  symbTx      = symbTx_[:,:,chIndex]

  # local oscillator (LO) parameters:
  FO          = 150e6                                                             # frequency offset
  Δf_lo       = freqGrid[chIndex]+FO                                              # downshift of the channel to be demodulated
  paramLO.Ns  = len(sigCh)

  sigLO       = basicLaserModel(paramLO)
  sigLO       = sigLO*np.exp(1j*2*π*Δf_lo*t) # add frequency offset

  # polarization multiplexed coherent optical received
  θsig        = π/3 # polarization rotation angle
  sigRx       = pdmCoherentReceiver(sigCh, sigLO, θsig, paramPD)

  return sigRx

In [ ]:
    ############## Filtro Casado e Compensação de CD ##############
def filtro_casado_cd(sigRx, paramTx, paramEDC):

  # Matched filtering
  paramPulse              = parameters()
  paramPulse.pulseType    = paramTx.pulseType
  paramPulse.SpS          = paramTx.SpS
  paramPulse.nFilterTaps  = paramTx.nFilterTaps
  paramPulse.pulseRollOff = paramTx.pulseRollOff

  pulse                   = pulseShape(paramPulse)

  pulse                   = pnorm(pulse)
  pulse                   = cp.asarray(pulse)
  sigRx                   = cp.asarray(sigRx)

  sigRx                   = firFilter(pulse, sigRx)
  sigRx                   = sigRx.get()

  # decimate to one sample per symbol
  paramDec                = parameters()
  paramDec.SpSin          = paramTx.SpS
  paramDec.SpSout         = 1
  sigRx                   = decimate(sigRx, paramDec)

  x                       = pnorm(sigRx)
  x                       = cp.asarray(x)

  return x

In [ ]:
    ############## Equalizadores ##############
def equalizador(x, paramTx):

  paramEq               = parameters()
  paramEq.nTaps         = 10
  paramEq.mu            = [1e-0, 1e-1]
  paramEq.M             = paramTx.M
  paramEq.constType     = paramTx.constType
  paramEq.N1            = 4000
  paramEq.N2            = 15000
  paramEq.alg           = 'cma-to-rde'
  paramEq.progBar       = True

  t_start1              = time.perf_counter()
  y_EQ                  = mimoAdaptEq(x.get(), paramEq)
  t_end1                = time.perf_counter()
  timeEqCPU             = t_end1 - t_start1

  t_start2              = time.perf_counter()
  y_EQ_gpu              = mimoAdaptEq(x, paramEq)
  t_end2                = time.perf_counter()
  timeEqGPU             = t_end2 - t_start2

  yEQ                   = y_EQ[0]

  # Parametros da Portadora de Fase
  paramCPR              = parameters()
  paramCPR.alg          = 'bps'
  paramCPR.M            = paramTx.M
  paramCPR.N            = 75
  paramCPR.B            = 64

  y_CPR                 = cpr(yEQ, paramCPR, symbTx_)

  # Equalizador Adaptativo
  paramEq.alg           = 'dd-lms'
  y_CPR_gpu             = cp.asarray(y_CPR)

  t_start3              = time.perf_counter()
  y_LMS, e, _           = mimoAdaptEq(y_CPR, paramEq)
  t_end3                = time.perf_counter()
  timeEqAdCPU           = t_end3 - t_start3

  t_start4              = time.perf_counter()
  y_LMS_gpu, e, _       = mimoAdaptEq(y_CPR_gpu, paramEq)
  t_end4                = time.perf_counter()
  timeEqAdGPU           = t_end4 - t_start4

  y_CPR_LMS             = cpr(y_LMS, paramCPR)

  return y_CPR, y_CPR_LMS, timeEqCPU, timeEqGPU, timeEqAdCPU, timeEqAdGPU

In [ ]:
    ############## Resultados ##############
def resultados(y_CPR_LMS, paramTx, symbTx_, Nsymb):

  discard = int(0.1 * y_CPR_LMS.shape[0])
  d = pnorm(symbTx_)
  ind = np.arange(discard, d.shape[0]-discard)
  # Remove a 3ª dimensão
  rx = y_CPR_LMS[ind,:]
  tx = d[ind,:]

  if rx.ndim == 3 and rx.shape[2] == 1:
        rx = rx[:, :, 0]
  if tx.ndim == 3 and tx.shape[2] == 1:
        tx = tx[:, :, 0]

  delay = finddelay(rx[:,1], tx[:,1])

  # Alinhar com symbolSync
  tx_sync = symbolSync(rx, tx, SpS=1, mode='amp')

  # Truncar sinais para comprimento comum
  min_len = min(rx.shape[0], tx_sync.shape[0])
  rx = rx[:min_len]
  tx = tx_sync[:min_len]

  delay = finddelay(rx[:,0], tx[:,0])

  rx = pnorm(rx)
  tx = pnorm(tx)
  BER, SER, SNR = fastBERcalc(rx, tx, paramTx.M, paramTx.constType)

  print('##########   Resultados da simulação - %s Nsymb   ##########\n'%Nsymb)
  print('      pol.X      pol.Y      ')
  print(' SER: %.2e,  %.2e'%(SER[0], SER[1]))
  print(' BER: %.2e,  %.2e'%(BER[0], BER[1]))
  print(' SNR: %.2f dB,  %.2f dB\n'%(SNR[0], SNR[1]))

In [ ]:
t_start = time.perf_counter()
##################### TRANSMISSOR #####################

# Transmitter parameters:
paramTx                 = parameters()
paramTx.M               = 64                                                    # order of the modulation format
paramTx.constType       = 'qam'                                                 # modulation scheme
paramTx.seed            = 42                                                    # seed
paramTx.Rs              = 32e9                                                  # symbol rate [baud]
paramTx.SpS             = 16                                                    # samples per symbol
paramTx.pulseType       = 'rrc'                                                 # pulse shaping filterN
paramTx.nFilterTaps     = 2048                                                  # number of pulse shaping filter coefficients
paramTx.pulseRollOff    = 0.01                                                  # RRC rolloff
paramTx.powerPerChannel = -2                                                    # power per WDM channel [dBm]
paramTx.nChannels       = 1                                                     # number of WDM channels
paramTx.Fc              = 193.1e12                                              # central optical frequency of the WDM spectrum
paramTx.laserLinewidth  = 100e3                                                 # laser linewidth in Hz
paramTx.wdmGridSpacing  = 37.5e9                                                # WDM grid spacing
paramTx.nPolModes       = 2                                                     # number of signal modes [2 for polarization multiplexed signals]

Fs                      = paramTx.Rs * paramTx.SpS                              # simulation sampling rate

## fiber parameters:
paramFiber              = parameters()
paramFiber.L            = 300                                                   # comprimento do enlace [km]
paramFiber.alpha        = 0.2                                                   # coeficiente de perdas [dB/Km]
paramFiber.D            = 17                                                    # parâmetro de dispersão [ps/nm/km]
paramFiber.Fs           = Fs                                                    # Frequência de amostragem do sinal [amostras/segundo]
paramFiber.Fc           = paramTx.Fc                                            # Frequência central do sinal [Hz]

# EDFA parameters:
paramEDFA               = parameters()
paramEDFA.G             = paramFiber.alpha * paramFiber.L                       # Ganho para compensar
paramEDFA.Fs            = Fs
paramEDFA.NF            = 7
paramEDFA.Fc            = paramTx.Fc

# generate CW laser LO field
paramLO                 = parameters()
paramLO.P               = 10                                                    # power in dBm
paramLO.lw              = 100e3                                                 # laser linewidth
paramLO.RIN_var         = 0                                                     # variancia do ruido
paramLO.Fs              = Fs

# Define os parâmetros do receptor coerente
paramPD                 = parameters()
paramPD.B               = paramTx.Rs
paramPD.Fs              = Fs
paramPD.ideal           = True

# CD compensation
paramEDC                = parameters()
paramEDC.L              = paramFiber.L
paramEDC.D              = paramFiber.D
paramEDC.Fc             = paramTx.Fc
paramEDC.Fs             = Fs

numberOfSymbols         = np.array([2e5])

timeEqCPU = np.zeros(len(numberOfSymbols))
timeEqGPU = np.zeros(len(numberOfSymbols))
timeEqAdCPU = np.zeros(len(numberOfSymbols))
timeEqAdGPU = np.zeros(len(numberOfSymbols))

for idx, Nsymb in enumerate(tqdm(numberOfSymbols)):

    paramTx.nBits       = int(np.log2(paramTx.M)*Nsymb)                         # total number of bits per polarization

    # generate WDM signal
    sigWDM_Tx, symbTx_, paramTx = simpleWDMTx(paramTx)

    sigCh               = canal_optico(sigWDM_Tx, paramEDFA, paramTx)           # função retorna o campo óptico complexo depois de passar pelo EDFA → AWGN → fibra linear.
    sigRx               = receptor_optico(sigCh, paramTx, paramPD)              # função retorna o sinal elétrico após o fotodetector balanceado após sigCh → LO → mixer → PD → sigRx
    x                   = filtro_casado_cd(sigRx, paramTx, paramEDC)            # função reorna o sinal após o filtro casado, a compensação de dispersão e decimação
    y_CPR, y_CPR_LMS, timeEqCPU, timeEqGPU, timeEqAdCPU, timeEqAdGPU            = equalizador(x, paramTx)         # função retorna os sinais equalizados apos o CMA → RDE → DD-LMS
    # Armazenando os tempos calculados em cada repetição
    timeEqCPU[idx]      = timeEqCPU
    timeEqGPU[idx]      = timeEqGPU
    timeEqAdCPU[idx]    = timeEqAdCPU
    timeEqAdGPU[idx]    = timeEqAdGPU
    resultados(y_CPR_LMS, paramTx, symbTx_, Nsymb)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

TypeError: Argument 'a' has incorrect type (expected cupy._core.core._ndarray_base, got numpy.ndarray)

In [ ]:
plt.plot(numberOfSymbols, timeEqCPU, '-o', linewidth=1.8, label='CPU')
plt.plot(numberOfSymbols, timeEqGPU, '-s', linewidth=1.8, label='GPU')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('Processing time (s)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

plt.figure()
plt.plot(numberOfSymbols, timeEqCPU/timeEqGPU,'-o', label='GPU speedup')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('GPU speedup ($\\times$)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

In [ ]:
plt.plot(numberOfSymbols, timeEqAdCPU, '-o', linewidth=1.8, label='CPU')
plt.plot(numberOfSymbols, timeEqAdGPU, '-s', linewidth=1.8, label='GPU')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('Processing time (s)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));

plt.figure()
plt.plot(numberOfSymbols, timeEqAdCPU/timeEqAdGPU,'-o', label='GPU speedup')
plt.xlabel('Length of the signal (number of samples)')
plt.ylabel('GPU speedup ($\\times$)')
plt.legend()
plt.grid()
plt.xlim(min(numberOfSymbols), max(numberOfSymbols));